CARREGAMENTO DA BASE DE DADOS:
__________________________________________________________________________________________________________________________________________________________________________________

In [3]:
# importando as bibliotecas necessárias para a análise de dados e visualização:
import pandas as pd
import matplotlib.pyplot as plt
 

# importando a base de dados diretamente do GitHub - repositório "Mini_projeto_varejo":
URL = 'https://github.com/mariahbarros/Mini_projeto_varejo/blob/main/base_varejo.csv?raw=true'

# transformando a base de dados em um dataframe:
df_original = pd.read_csv(URL, sep=';', encoding='utf-8')

# criando uma cópia do dataframe original para trabalhar a análise sem alterar os dados originais:
df = df_original.copy()

# formatação para exibir os números em porcentagem com duas casas decimais:
pd.set_option('display.float_format', '{:,.2f}%'.format)


ANÁLISE PRIMÁRIA DO DATAFRAME:
__________________________________________________________________________________________________________________________________________________________________________________

In [4]:
# verificando as informações gerais sobre a base de dados:
print('====== DIMENSÕES DA BASE DE DADOS =====')
print(f'\nBase carregada com {df.shape[0]} linhas e {len(df.columns)} colunas.')                    # verificando o número de linhas e colunas do dataframe

print('\n===== ANALISANDO AS PRIMEIRAS E ÚLTIMAS LINHAS DOS DADOS =====')
print(f'\nInformações sobre as primeiras 5 linhas: \n{df.head(5).to_string()}')                     # verificando as primeiras 5 linhas do dataframe
print(f'\nInformações sobre as últimas 5 linhas: \n{df.tail(5).to_string()}')                       # verificando as últimas 5 linhas do dataframe

print('\n===== INFORMAÇÕES GERAIS =====')
print(f'\nInformações gerais sobre os dados:')                                                      # verificando as informações gerais sobre o dataframe
df.info()

print('\n===== INFORMAÇÕES SOBRE COLUNAS =====')
print(f'\nVerificando a distribuição de valores únicos por coluna: \n{df.nunique().to_string()}')   # verificando a distribuição de valores únicos por coluna
 
print('\n===== INFORMAÇÕES SOBRE VALORES DUPLICADOS =====')
print(f'\nVerificando duplicatas: {df.duplicated().sum()}')                                         # verificando a existência de valores duplicados no dataframe
 

====== DIMENSÕES DA BASE DE DADOS =====

Base carregada com 830000 linhas e 14 colunas.

===== ANALISANDO AS PRIMEIRAS E ÚLTIMAS LINHAS DOS DADOS =====

Informações sobre as primeiras 5 linhas: 
         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT               PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS  REFRIGERANTE GUARANA          NaN          NaN          NaN          NaN
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   REFRIGERANTE OUTROS          NaN          NaN          NaN          NaN
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE       LENCO UMEDECIDO          NaN          NaN          NaN          NaN
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS               ABACAXI          NaN          NaN          NaN          NaN
4  01/02/2019   1000    534         M      4 

TRATAMENTO DOS DADOS:
__________________________________________________________________________________________________________________________________________________________________________________

In [5]:
# 1. transformando a coluna 'DATA' em formato datetime:
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y')

# 1.1. verificando o período analisado no dataframe:
print("\nVerificando o período analisado:")
print(f"Período em análise: {df['DATA'].min().date().strftime('%d/%m/%Y')} até {df['DATA'].max().date().strftime('%d/%m/%Y')}")




Verificando o período analisado:
Período em análise: 04/01/2019 até 08/12/2022


In [6]:
# 2. transformando as colunas do tipo 'int64' (CO_ID/CL_ID/CL_EC/PR_ID) para "category", para otimizar a memória e facilitar a análise dos dados:

# 2.1. inicialmente, vamos criar um dicionario para mapear os valores da coluna CL_EC (estado civil) para uma representação mais legível:
legenda_CL_EC = {
    1: 'casado/uniao estavel',
    2: 'divorciado',
    3: 'separado',
    4: 'solteiro',
    5: 'viuvo',

}

df['CL_EC'] = df['CL_EC'].replace(legenda_CL_EC)            # usando o .replace para substituir os valores

# 2.2. criando uma lista com o nome de todas as colunas que serão alteradas para conversão em loop:
colunas_para_conversao = ['CO_ID', 'CL_ID', 'CL_EC', 'PR_ID']          
for coluna in colunas_para_conversao:       
    if coluna in df.columns: 
        df[coluna] = df[coluna].astype('category')
 
# verificação:
print('Verificando a alteração dos dados da coluna CL_EC:')

substituicao_EC = df["CL_EC"].sample(10).tolist()           # verificando 10 valores aleatórios da coluna CL_EC para confirmar a substituição
print(substituicao_EC)
 



Verificando a alteração dos dados da coluna CL_EC:
['divorciado', 'divorciado', 'solteiro', 'separado', 'solteiro', 'solteiro', 'casado/uniao estavel', 'solteiro', 'solteiro', 'casado/uniao estavel']


In [7]:
# 3. removendo as colunas vazias (10, 11, 12 e 13):
df = df.dropna(axis=1, how='all')

print(f"\nVerificando as colunas remanescentes: \n{df.columns.tolist()}")
  


Verificando as colunas remanescentes: 
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME']


In [8]:
# 4. tratando os dados duplicados:

# 4.1. removendo definitivamente as linhas duplicadas, mantendo a primeira ocorrência:
df.drop_duplicates(keep='first', inplace=True)

df = df.reset_index(drop=True)          # renumerando as linhas após a remoção das colunas vazias e duplicatas


In [9]:
# verificou-se a existência do valor '#N/D' na coluna PR_CAT (ocorrência de 0,44%)

# verificando se o mesmo valor '#N/D' também aparece na coluna PR_NOME:
ocorrencia = (df['PR_NOME'] == '#N/D').sum()
print(f'\nO valor "#N/D" aparece exatamente {ocorrencia} vezes na coluna "PR_NOME".')

# substituindo o texto '#N/D' por 'Dado Ausente' nas colunas PR_CAT e PR_NOME:
df['PR_CAT'] = df['PR_CAT'].replace('#N/D', 'Dado Ausente')
df['PR_NOME'] = df['PR_NOME'].replace('#N/D', 'Dado Ausente')

print("==================================================")

# verificação:
print("Frequência atual do valor 'Dado Ausente':")
print(f'PR_CAT: {df["PR_CAT"].value_counts().loc["Dado Ausente"]}')
print(f'PR_NOME: {df["PR_NOME"].value_counts().loc["Dado Ausente"]}')



O valor "#N/D" aparece exatamente 3228 vezes na coluna "PR_NOME".
Frequência atual do valor 'Dado Ausente':
PR_CAT: 3228
PR_NOME: 3228


In [10]:
# verificação final após tratamento dos dados:
print('====== INFORMAÇÕES GERAIS APÓS O TRATAMENTO DOS DADOS ======')

print(f'\nNovo formato do dataframe após remoção das colunas vazias e linhas duplicadas: {df.shape[0]} linhas e {len(df.columns)} colunas.')
print(f'\nInformações gerais sobre os dados após tratamento:')
df.info()
print(f'\nVerificando a distribuição de valores únicos por coluna: \n{df.nunique().to_string()}')
print(f'\nEstatísticas descritivas gerais da coluna "CL_FHL" após o tratamento: \n{df["CL_FHL"].describe().map('{:.2f}'.format)}')
print(f'\nVerificando a existência de valores nulos após o tratamento: \n{df.isnull().sum().to_string()}')
print(f'\nVerificando valores duplicados após o tratamento: \n{df.duplicated().sum()}')



====== INFORMAÇÕES GERAIS APÓS O TRATAMENTO DOS DADOS ======

Novo formato do dataframe após remoção das colunas vazias e linhas duplicadas: 733447 linhas e 10 colunas.

Informações gerais sobre os dados após tratamento:
<class 'pandas.DataFrame'>
RangeIndex: 733447 entries, 0 to 733446
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       733447 non-null  datetime64[us]
 1   CO_ID      733447 non-null  category      
 2   CL_ID      733447 non-null  category      
 3   CL_GENERO  733447 non-null  str           
 4   CL_EC      733447 non-null  category      
 5   CL_FHL     733447 non-null  int64         
 6   CL_SEG     733447 non-null  str           
 7   PR_ID      733447 non-null  category      
 8   PR_CAT     733447 non-null  str           
 9   PR_NOME    733447 non-null  str           
dtypes: category(4), datetime64[us](1), int64(1), str(4)
memory usage: 38.6 MB

Verificando a distribui

ANÁLISE EXPLORATÓRIA DOS DADOS:
__________________________________________________________________________________________________________________________________________________________________________________

In [11]:
# VERIFICANDO O PERFIL GERAL DO CLIENTE (ANÁLISE POR GÊNERO E CLASSE ECONÔMICA):
print('\n=====PERFIL DO CLIENTE POR GÊNERO:=====')
print((df['CL_GENERO'].value_counts(normalize=True) * 100).to_string(name=False))

# VERIFICANDO O ESTADO CIVIL DOS CLIENTES:
print('\n=====PERFIL DO CLIENTE POR ESTADO CIVIL:=====')
print((df['CL_EC'].value_counts(normalize=True) * 100).to_string(name=False))

# VERIFICANDO SE OS CLIENTES POSSUEM FILHOS:
print('\n=====PERFIL DO CLIENTE POR FILHOS:=====')
print((df['CL_FHL'].value_counts(normalize=True) * 100).round(2).to_string(name=False))

# VERIFICANDO A CLASSE ECONÔMICA DOS CLIENTES:
print('\n=====PERFIL DO CLIENTE POR CLASSE ECONÔMICA:=====')
print((df['CL_SEG'].value_counts(normalize=True) * 100).to_string(name=False))

# VERIFICANDO OS PRODUTOS MAIS CONSUMIDOS:
print('\n=====CATEGORIAS DE PRODUTOS MAIS CONSUMIDAS:=====')
print((df['PR_CAT'].value_counts(normalize=True) * 100).to_string(name=False))

# VERIFICANDO OS 10 PRODUTOS MAIS CONSUMIDOS:
print('\n=====OS 10 PRODUTOS MAIS CONSUMIDOS:=====')
print((df['PR_NOME'].value_counts(normalize=True) * 100).head(10).to_string(name=False))

# VERIFICANDO AS INFORMAÇÕES SOBRE AS COMPRAS:
print('\n=====INFORMAÇÕES SOBRE AS COMPRAS DOS CLIENTES:=====')
print(df.groupby('CO_ID').size().describe().map('{:.2f}'.format))



=====PERFIL DO CLIENTE POR GÊNERO:=====
CL_GENERO
F   52.14%
M   47.86%

=====PERFIL DO CLIENTE POR ESTADO CIVIL:=====
CL_EC
separado               25.78%
solteiro               24.43%
divorciado             23.49%
casado/uniao estavel   23.45%
viuvo                   2.85%

=====PERFIL DO CLIENTE POR FILHOS:=====
CL_FHL
0   52.49%
2   12.84%
3   12.60%
1   12.39%
4    9.69%

=====PERFIL DO CLIENTE POR CLASSE ECONÔMICA:=====
CL_SEG
B   63.88%
C   27.99%
A    8.14%

=====CATEGORIAS DE PRODUTOS MAIS CONSUMIDAS:=====
PR_CAT
ALIMENTOS      52.38%
HIGIENE        18.77%
LIMPEZA        17.54%
BEBIDAS         5.22%
PET             3.89%
ACESSORIOS      1.75%
Dado Ausente    0.44%

=====OS 10 PRODUTOS MAIS CONSUMIDOS:=====
PR_NOME
PRESUNTO COZIDO      1.73%
SARDINHA             0.90%
BANANA               0.89%
ESCOVA DE DENTE      0.89%
GEL                  0.89%
PAPINHA INFANTIL     0.89%
MODELADOR            0.89%
CERA                 0.89%
LIMPADOR PERFUMADO   0.89%
CEBOLA               0.8

In [12]:
# VERIFICANDO AS VENDAS (TRANSAÇÕES) POR DIA DA SEMANA E POR MÊS:

print('\n===== TRANSAÇÕES POR DIA DA SEMANA =====')

transacoes_dias_semana = df.groupby(df['DATA'].dt.strftime('%A').str.capitalize())['CO_ID'].nunique()    # agrupando os dias da semana com os códigos de compra (transações)

traducao_dias = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

transacoes_dias_semana = transacoes_dias_semana.rename(index=traducao_dias).sort_values(ascending=False)   # traduzindo os dias da semana e organizando a ordem

print(transacoes_dias_semana)    

print('\n===== TRANSAÇÕES POR MÊS =====')
 
transacoes_por_mes = df.groupby(df['DATA'].dt.strftime('%m - %B'))['CO_ID'].nunique()           # agrupando os meses com os códigos de compra (transações)
 
print(transacoes_por_mes)    


===== TRANSAÇÕES POR DIA DA SEMANA =====
DATA
Quarta-feira     3450
Sexta-feira      2905
Domingo          2894
Segunda-feira    2470
Quinta-feira     2318
Terça-feira      2294
Sábado           2140
Name: CO_ID, dtype: int64

===== TRANSAÇÕES POR MÊS =====
DATA
01 - January      1859
02 - February     1699
03 - March        1442
04 - April        1478
05 - May          1839
06 - June         1503
07 - July         1534
08 - August       1463
09 - September    1536
10 - October      1645
11 - November      921
12 - December     1552
Name: CO_ID, dtype: int64


In [13]:
# VERIFICANDO AS TRANSAÇÕES REALIZADAS POR ANO:

print('\n===== TRANSAÇÕES POR ANO =====')
resumo = df.groupby(df['DATA'].dt.year)['CO_ID'].nunique().to_frame(name='Transações Totais')        # contabilizando quantas transações ocorrem em relação à coluna 'data' (ano)

resumo['Variacao_Anual'] = resumo['Transações Totais'].pct_change().map('{:+.2f}%'.format).fillna('-')

print(resumo)  


===== TRANSAÇÕES POR ANO =====
      Transações Totais Variacao_Anual
DATA                                  
2019               4483          +nan%
2020               4794         +0.07%
2021               5505         +0.15%
2022               3689         -0.33%


In [14]:
# COMPARANDO AS VENDAS (TRANSAÇÕES) MENSAIS ANO A ANO:

print('\n===== COMPARAÇÃO DE COMPRAS (TRANSAÇÕES) MÊS A MÊS (2019-2022) =====')

print(df.groupby([df['DATA'].dt.strftime('%m - %B'), df['DATA'].dt.year])['CO_ID'].nunique().unstack(fill_value=0))    # agrupando por mês e ano



===== COMPARAÇÃO DE COMPRAS (TRANSAÇÕES) MÊS A MÊS (2019-2022) =====
DATA            2019  2020  2021  2022
DATA                                  
01 - January     171   506   616   566
02 - February    402   434   433   430
03 - March       257   439   382   364
04 - April       410   387   363   318
05 - May         711   366   333   429
06 - June        493   264   445   301
07 - July        274   215   599   446
08 - August      420   384   386   273
09 - September   436   471   594    35
10 - October     242   633   711    59
11 - November    151   384   231   155
12 - December    516   311   412   313


In [15]:
# CONTABILIZANDO QUANTAS TRANSACOES FORAM REALIZADAS ENTRE OS CLIENTES COM E SEM FILHOS:

df['Tem_Filhos'] = df['CL_FHL'].apply(lambda x: 'Sem Filhos' if x == 0 else 'Com Filhos')   # criando uma coluna para classificar os clientes em 'Com Filhos' e 'Sem Filhos'

print('===== ANÁLISE DE TRANSAÇÕES DE ACORDO COM A EXISTÊNCIA DE FILHOS =====')

total_transacoes = df.groupby('Tem_Filhos')['CO_ID'].nunique()                              # agrupando as informações de transações por clientes com e sem filhos

print(total_transacoes)

percentual_transacoes = (df.groupby('Tem_Filhos')['CO_ID'].nunique() / df['CO_ID'].nunique() * 100).round(2)    # calculando o percentual

print('\n===== PERCENTUAL (%): =====')
print(percentual_transacoes)

===== ANÁLISE DE TRANSAÇÕES DE ACORDO COM A EXISTÊNCIA DE FILHOS =====
Tem_Filhos
Com Filhos    8736
Sem Filhos    9735
Name: CO_ID, dtype: int64

===== PERCENTUAL (%): =====
Tem_Filhos
Com Filhos   47.30%
Sem Filhos   52.70%
Name: CO_ID, dtype: float64


In [16]:
# AGORA, VERIFICANDO A QUANTIDADE DE PRODUTOS ADQUIRIDOS POR CLIENTES COM E SEM FILHOS:

print('===== ANÁLISE DE PRODUTOS ADQUIRIDOS DE ACORDO COM A EXISTÊNCIA DE FILHOS =====')

# .size() conta o total de linhas (produtos) para cada grupo
total_produtos = df.groupby('Tem_Filhos').size()                                        # contabilizando a quantidade total de produtos adquiridos por clientes com e sem filhos

print('TOTAL DE PRODUTOS COMPRADOS:')
print(total_produtos)

percentual_produtos = (df.groupby('Tem_Filhos').size() / len(df) * 100)                 # calculando o %

print('\n===== PERCENTUAL (%): =====')
print(percentual_produtos)


===== ANÁLISE DE PRODUTOS ADQUIRIDOS DE ACORDO COM A EXISTÊNCIA DE FILHOS =====
TOTAL DE PRODUTOS COMPRADOS:
Tem_Filhos
Com Filhos    348461
Sem Filhos    384986
dtype: int64

===== PERCENTUAL (%): =====
Tem_Filhos
Com Filhos   47.51%
Sem Filhos   52.49%
dtype: float64


In [17]:
# COMPARANDO AS VENDAS (TRANSAÇÕES) POR QUANTIDADE DE FILHOS:

print('===== ANÁLISE DE TRANSAÇÕES POR QUANTIDADE DE FILHOS =====')

analise_filhos = df.groupby('CL_FHL')['CO_ID'].nunique().reset_index(name='Total_Transacoes')       # agrupando dados das transações por quantidade de filhos

total_geral_compras = analise_filhos['Total_Transacoes'].sum()                                      # calculando o % de com ou sem filhos em relação ao total de transações
analise_filhos['%_DO_TOTAL'] = (analise_filhos['Total_Transacoes'] / total_geral_compras * 100)

print(analise_filhos.to_string(index=False))

===== ANÁLISE DE TRANSAÇÕES POR QUANTIDADE DE FILHOS =====
 CL_FHL  Total_Transacoes  %_DO_TOTAL
      0              9735      52.70%
      1              2250      12.18%
      2              2394      12.96%
      3              2289      12.39%
      4              1803       9.76%


In [ ]:
# ANALISANDO A QUANTIDADE E A MÉDIA DE PRODUTOS ADQUIRIDOS POR QUANTIDADE DE FILHOS:

print('===== ANÁLISE DE PRODUTOS E MÉDIAS POR QUANTIDADE DE FILHOS =====')


analise_filhos = df.groupby('CL_FHL').agg(                          # agrupando a quantidade de filhos e calculando o total de produtos e total de transações
    Total_Produtos=('CO_ID', 'size'),
    Total_Compras=('CO_ID', 'nunique')
).reset_index()

total_geral_produtos = analise_filhos['Total_Produtos'].sum()                       # calculando o total geral de produtos e o % 
analise_filhos['%_DO_TOTAL'] = ((analise_filhos['Total_Produtos'] / total_geral_produtos) * 100).round(2)

analise_filhos['Media_Produtos_Por_Carrinho'] = (analise_filhos['Total_Produtos'] / analise_filhos['Total_Compras']).round(2).map('{:.2f}'.format)   # calculando a média e retira o % da média

print(analise_filhos[colunas_ordenadas].to_string(index=False))

===== ANÁLISE DE PRODUTOS E MÉDIAS POR QUANTIDADE DE FILHOS =====
 CL_FHL  Total_Produtos  %_DO_TOTAL Media_Produtos_Por_Carrinho
      0          384986      52.49%                       39.55
      1           90845      12.39%                       40.38
      2           94168      12.84%                       39.34
      3           92407      12.60%                       40.37
      4           71041       9.69%                       39.40


In [28]:
# CALCULANDO OS 5 PRODUTOS MAIS CONSUMIDOS ENTRE CLINTES COM E SEM FILHOS:

print('\n===== OS 5 PRODUTOS MAIS CONSUMIDOS ENTRE OS CLIENTES COM E SEM FILHOS =====')

df_produtos_validos = df[df['PR_NOME'] != 'valor ausente']                                                      # descartando os produtos "Valor Ausente"

top_5_perfil = df_produtos_validos.groupby(['Tem_Filhos', 'PR_NOME']).size()                                    # agrupando por perfil e nome do produto
top_5_final = top_5_perfil.groupby(level=0, group_keys=False).nlargest(5).reset_index(name='QTD_VENDAS')

total_perfil = top_5_final.groupby('Tem_Filhos')['QTD_VENDAS'].transform('sum')                                 # calculando o %
top_5_final['%_DO_PERFIL'] = ((top_5_final['QTD_VENDAS'] / total_perfil) * 100)

print(top_5_final.to_string(index=False))



===== OS 5 PRODUTOS MAIS CONSUMIDOS ENTRE OS CLIENTES COM E SEM FILHOS =====
Tem_Filhos            PR_NOME  QTD_VENDAS  %_DO_PERFIL
Com Filhos    PRESUNTO COZIDO        6051       32.54%
Com Filhos  LIMPADOR MULTIUSO        3178       17.09%
Com Filhos        BATATA DOCE        3131       16.84%
Com Filhos    ESCOVA DE DENTE        3119       16.77%
Com Filhos            IOGURTE        3119       16.77%
Sem Filhos    PRESUNTO COZIDO        6668       32.39%
Sem Filhos           SARDINHA        3534       17.16%
Sem Filhos             CEBOLA        3470       16.85%
Sem Filhos LIMPADOR PERFUMADO        3464       16.82%
Sem Filhos   PAPINHA INFANTIL        3453       16.77%


In [29]:
# SALVANDO O DATAFRAME APÓS TRATAMENTO EM UM ARQUIVO CSV    
df.to_csv('base-varejo_df.limpo.csv', index=False, sep=';', encoding='utf-8-sig')

print('Arquivo CSV salvo com sucesso!')

Arquivo CSV salvo com sucesso!
